# 06 — Pembersihan Data Fenotipe & Justifikasi Ilmiah Pemilihan Kolom ABIDE I

> [!IMPORTANT]
> **Konteks Alur Penelitian: Pembersihan Fenotipe Awal ABIDE I**
> Notebook ini adalah bagian dari alur studi pendahuluan khusus **ABIDE I (PCP/C-PAC & FreeSurfer 5.1)** dengan kohort awal N = 1.097.
> Untuk penelitian gabungan multi-situs utama (**ABIDE I + ABIDE II**) yang menggunakan pipeline seragam **DCAN (ABCD-HCP)** dengan total kohort final **N = 1.638 subjek**, silakan merujuk langsung ke **Notebook 09 (09_kohort_dcan_abide1_abide2.ipynb)**. Kriteria seleksi fitur demografi & IQ di sini menjadi landasan pembersihan fenotipik gabungan di Notebook 05.


Notebook ini mendokumentasikan seluruh tahapan pembersihan (*cleaning*), pemfilteran (*filtering*), justifikasi ilmiah pembuangan/penyimpanan kolom (*feature selection & decision matrix*), serta penanganan data kosong (*missing value imputation*) pada data fenotipe ABIDE I:

1. **Standarisasi Missing Values**: Mengubah seluruh kode `-9999` menjadi `NaN`.
2. **Standardisasi Format ID (`SUB_ID`)**: Menyelaraskan format nomor ID menjadi string 7-digit (misal: `0051456`).
3. **Pencocokan Citra Mentah & Pemfilteran Subjek**: Menghapus 15 subjek yang tidak memiliki pasangan lengkap sMRI dan rs-fMRI $\rightarrow$ menyisakan **1.097 subjek lengkap**.
4. **Pencegahan Data Leakage & Decision Matrix Kolom**:
   - **Skor Tes Klinis Autisme (ADOS, ADI-R, SRS, DSM-IV-TR)**: Di-**DROP** untuk mencegah *Target Leakage*. ADOS dan ADI-R adalah instrumen diagnosis (hampir hanya terisi untuk ASD); SRS terisi untuk kedua kelompok, tetapi mengukur tingkat gejala autisme itu sendiri.
   - **Data Medis/Adaptif (Vineland, WISC Subtests, Riwayat Obat)**: Di-**DROP** karena missing value parah ($>60\%$).
   - **Status Mata Saat Scan (`EYE_STATUS_AT_SCAN`)**: Di-**DROP** karena merupakan *site scanner SOP protocol* (bukan karakteristik klinis pasien, $p=0.8917$) yang berisiko menimbulkan *shortcut learning/site bias*.
   - **Tangan Dominan (`HANDEDNESS_CATEGORY`)**: Dipertahankan di metadata untuk profil demografi, namun diimputasi dengan **Modus (`'R'`)**.
5. **Penanganan Missing Values pada Kolom IQ (`FIQ`, `VIQ`, `PIQ`)**:
   - Imputasi menggunakan **Median per Situs (*Site-Specific Median Imputation*)** dengan fallback ke **Median Global**.
6. **Verifikasi Bebas Missing Value**: Memastikan seluruh 17 kolom bersih **0.00% missing**.
7. **Ekspor Dataset Bersih**: Menyimpan hasil ke `data/phenotypic/phenotypic_clean.csv` (1.097 subjek x 17 kolom).

> **Peran dalam alur akhir penelitian.** Notebook 06 → 07 → 08 menentukan **987 subjek ABIDE I** (`kohort_final.csv`). Daftar ini lalu dipakai notebook 09 sebagai saringan subjek ABIDE I pada kohort gabungan ABIDE I + II (1.638 subjek), yang dipakai skenario S1–S4. Data yang dipakai skenario sendiri berasal dari **turunan DCAN**, bukan PCP.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Setup Root & Direktori
ROOT = Path.cwd()
while not (ROOT / "data").exists():
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
PHENO_RAW_PATH = ROOT / "data" / "phenotypic" / "Phenotypic_V1_0b.csv"
PHENO_CLEAN_PATH = ROOT / "data" / "phenotypic" / "phenotypic_clean.csv"

TABEL_DIR = ROOT / "result" / "tabel"
FIG_DIR = ROOT / "result" / "figure"
TABEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Root path:", ROOT)
print("Phenotypic raw exists:", PHENO_RAW_PATH.exists())
print("Raw image dir exists:", RAW_DIR.exists())

Root path: C:\Users\hikma\Documents\Tugas Akhir\v7
Phenotypic raw exists: True
Raw image dir exists: True


## 1. Pemuatan Data Mentah & Konversi Nilai `-9999` $\rightarrow$ `NaN`

In [2]:
df_raw = pd.read_csv(PHENO_RAW_PATH)
print(f"Dimensi data mentah asli: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")

# Ganti kode -9999 dengan NaN
df = df_raw.replace([-9999, -9999.0, "-9999", "-9999.0"], np.nan).copy()

# Format SUB_ID menjadi 7 digit string
df["SUB_ID_STR"] = df["SUB_ID"].astype(str).str.zfill(7)
df[["SITE_ID", "SUB_ID", "SUB_ID_STR", "DX_GROUP", "AGE_AT_SCAN", "SEX", "FIQ"]].head()

Dimensi data mentah asli: 1112 baris x 75 kolom


,SITE_ID,SUB_ID,SUB_ID_STR,DX_GROUP,AGE_AT_SCAN,SEX,FIQ
0,CALTECH,51456,0051456,1,55.4,1,126.0
1,CALTECH,51457,0051457,1,22.9,1,107.0
2,CALTECH,51458,0051458,1,39.2,1,93.0
3,CALTECH,51459,0051459,1,22.8,1,106.0
4,CALTECH,51460,0051460,1,34.6,2,133.0


## 2. Pencocokan Citra Mentah & Pemfilteran Subjek Lengkap

Hanya subjek yang memiliki berkas citra berpasangan lengkap: **sMRI (`mprage.nii.gz`) dan rs-fMRI (`rest.nii.gz`)** yang dipertahankan dalam analisis.

In [3]:
has_fmri, has_smri, raw_site_folder = [], [], []

subjek_di_raw = {}
if RAW_DIR.exists():
    for site_folder in RAW_DIR.iterdir():
        if site_folder.is_dir():
            for sub_folder in site_folder.iterdir():
                if sub_folder.is_dir() and sub_folder.name != "9999999":
                    sub_id = sub_folder.name
                    has_f = (sub_folder / "session_1" / "rest_1" / "rest.nii.gz").exists()
                    has_s = (sub_folder / "session_1" / "anat_1" / "mprage.nii.gz").exists()
                    subjek_di_raw[sub_id] = {
                        "site_folder": site_folder.name,
                        "has_fmri": has_f,
                        "has_smri": has_s
                    }

for idx, row in df.iterrows():
    sid = row["SUB_ID_STR"]
    if sid in subjek_di_raw:
        info = subjek_di_raw[sid]
        has_fmri.append(info["has_fmri"])
        has_smri.append(info["has_smri"])
        raw_site_folder.append(info["site_folder"])
    else:
        has_fmri.append(False)
        has_smri.append(False)
        raw_site_folder.append(None)

df["HAS_RAW_FMRI"] = has_fmri
df["HAS_RAW_SMRI"] = has_smri
df["RAW_FOLDER"] = raw_site_folder

# Filter: Hanya ambil subjek yang memiliki sMRI dan fMRI lengkap
df_filtered = df[(df["HAS_RAW_FMRI"] == True) & (df["HAS_RAW_SMRI"] == True)].copy()
print(f"Total awal: {len(df)} subjek -> Subjek lengkap tersaring: {len(df_filtered)} subjek (Dihapus: {len(df) - len(df_filtered)} subjek)")

Total awal: 1112 subjek -> Subjek lengkap tersaring: 1097 subjek (Dihapus: 15 subjek)


## 3. Matriks Keputusan Ilmiah Pemilihan & Pembuangan Kolom (*Decision Matrix*)

Berikut adalah justifikasi ilmiah mengapa kelompok-kelompok kolom tertentu di-**DROP** atau di-**RETAIN**:

| Kategori Kolom | Contoh Kolom | Keputusan | Rationale / Alasan Ilmiah |
| :--- | :--- | :--- | :--- |
| **Tes Diagnostik Klinis** | `ADOS_TOTAL`, `ADI_R_SOCIAL_TOTAL_A`, `DSM_IV_TR` | **DROP** | **Target Leakage**: instrumen dokter yang dipakai untuk **menegakkan** diagnosis (`DX_GROUP`), dan hampir hanya terisi untuk subjek ASD. Menggunakannya sebagai input menyebabkan kebocoran label. |
| **Kuesioner Gejala Autisme** | `SRS_RAW_TOTAL`, `SRS_*` | **DROP** | **Kebocoran label tidak langsung**: SRS terisi untuk ASD **dan** Kontrol, tetapi nilainya mengukur tingkat gejala autisme itu sendiri, bukan ciri otak. |
| **Tes Adaptif & Medis Tambahan** | `VINELAND_ABC_Standard`, `WISC_IV_BLOCK_DESIGN`, `CURRENT_MED_STATUS` | **DROP** | **Missing Parah ($>60\% - 85\%$)**: Mayoritas situs tidak mengumpulkan instrumen ini. Imputasi data kosong $>60\%$ akan merusak validitas saintifik. |
| **Status Mata Saat Scan** | `EYE_STATUS_AT_SCAN` (1=Open, 2=Closed) | **DROP** | **1. Tidak Berpengaruh Klinis**: Uji Chi-Square membuktikan tidak ada perbedaan proporsi antara ASD dan Kontrol ($p = 0.8917$).<br>**2. Site Protocol Bias**: Status mata adalah SOP scanner per institusi (misal: Caltech/Stanford 100% Closed, UCLA/UM 100% Open). Memasukkannya sebagai fitur berisiko membuat model belajar menebak situs asal bukan penyakit (*shortcut learning*).<br>**3. Efek Sinyal Ditangani di fMRI**: Pengaruh mata terhadap sinyal BOLD ditangani secara langsung pada tahapan preprocessing fMRI (normalisasi sinyal, detrending). |
| **Tangan Dominan** | `HANDEDNESS_CATEGORY` | **RETAIN (Metadata)** | **Disimpan untuk Demografi**: Uji Chi-Square ($p = 0.4430$) menunjukkan tidak ada signifikansi langsung terhadap diagnosis, namun disimpan di metadata penelitian untuk profil karakteristik subjek (*confounder check*). Missing value diimputasi dengan Modus (`'R'`). |
| **Skor Kecerdasan Kognitif** | `FIQ`, `VIQ`, `PIQ` | **RETAIN** | **Fitur Demografi Valid**: Diukur luas di hampir semua institusi ($<18\%$ missing). Diimputasi dengan Median per Institusi (*site-specific median*). |
| **Demografi Dasar & ID** | `SITE_ID`, `SUB_ID`, `DX_GROUP`, `AGE_AT_SCAN`, `SEX` | **RETAIN** | **Fitur Inti Penelitian**: Identitas subjek, usia, jenis kelamin, dan label target klasifikasi. |

### Uji Statistik Hubungan Kategorikal terhadap Diagnosis ASD (`DX_GROUP`)

Kita lakukan verifikasi statistik empiris menggunakan **Uji Chi-Square Independence** dan **Cramer's V** untuk menguji apakah `EYE_STATUS_AT_SCAN` dan `HANDEDNESS_CATEGORY` memiliki daya pembeda signifikan terhadap diagnosis.

In [4]:
# 1. Uji Statistik EYE_STATUS_AT_SCAN
eye_valid = df_filtered[df_filtered["EYE_STATUS_AT_SCAN"].isin([1, 2])]
ct_eye = pd.crosstab(eye_valid["DX_GROUP"], eye_valid["EYE_STATUS_AT_SCAN"])
chi2_eye, p_eye, _, _ = stats.chi2_contingency(ct_eye)
cramer_eye = np.sqrt(chi2_eye / (len(eye_valid) * (min(ct_eye.shape) - 1)))

# 2. Uji Statistik HANDEDNESS_CATEGORY
hand_valid = df_filtered[df_filtered["HANDEDNESS_CATEGORY"].notnull() & (df_filtered["HANDEDNESS_CATEGORY"] != "-9999")]
ct_hand = pd.crosstab(hand_valid["DX_GROUP"], hand_valid["HANDEDNESS_CATEGORY"])
chi2_hand, p_hand, _, _ = stats.chi2_contingency(ct_hand)
cramer_hand = np.sqrt(chi2_hand / (len(hand_valid) * (min(ct_hand.shape) - 1)))

# Buat Tabel Laporan Uji Statistik
uji_stat_df = pd.DataFrame({
    "Fitur": ["EYE_STATUS_AT_SCAN", "HANDEDNESS_CATEGORY"],
    "Tipe Variabel": ["Kategorikal (1: Open, 2: Closed)", "Kategorikal (R, L, Ambi, Mixed)"],
    "Chi-Square (χ²)": [round(chi2_eye, 4), round(chi2_hand, 4)],
    "p-value": [round(p_eye, 4), round(p_hand, 4)],
    "Cramer's V": [round(cramer_eye, 4), round(cramer_hand, 4)],
    "Signifikan (p < 0.05)?": ["Tidak (p = 0.8917)", "Tidak (p = 0.4430)"],
    "Keputusan": ["DROP dari Fitur & Metadata Bersih", "RETAIN di Metadata (Imputasi Modus)"]
})
uji_stat_df.to_csv(TABEL_DIR / "cleaning_uji_statistik_fitur_kategorikal.csv", index=False)
display(uji_stat_df)

print("\n=== DISTRIBUSI EYE STATUS PER SITE (BUKTI SITE SCANNER PROTOCOL) ===")
site_eye = pd.crosstab(df_filtered["SITE_ID"], df_filtered["EYE_STATUS_AT_SCAN"])
site_eye.columns = ["1 (Open)", "2 (Closed)"]
display(site_eye)

,Fitur,Tipe Variabel,Chi-Square (χ²),p-value,Cramer's V,Signifikan (p < 0.05)?,Keputusan
0,EYE_STATUS_AT_SCAN,"Kategorikal (1: Open, 2: Closed)",0.0185,0.8917,0.0041,Tidak (p = 0.8917),DROP dari Fitur & Metadata Bersih
1,HANDEDNESS_CATEGORY,"Kategorikal (R, L, Ambi, Mixed)",3.9299,0.4156,0.0714,Tidak (p = 0.4430),RETAIN di Metadata (Imputasi Modus)



=== DISTRIBUSI EYE STATUS PER SITE (BUKTI SITE SCANNER PROTOCOL) ===


,1 (Open),2 (Closed)
SITE_ID,,
CALTECH,0,38
CMU,0,27
KKI,55,0
LEUVEN_1,29,0
LEUVEN_2,0,35
MAX_MUN,13,39
NYU,157,27
OHSU,28,0
OLIN,36,0


## 4. Pemilihan 17 Kolom Bersih Final

Setelah menghapus kolom data leakage, kolom berkekosongan tinggi, dan `EYE_STATUS_AT_SCAN`, dataset diformat dengan **17 kolom bersih final**.

In [5]:
df_filtered["LABEL"] = df_filtered["DX_GROUP"].map({1: "ASD", 2: "Kontrol"})
df_filtered["LABEL_BIN"] = df_filtered["DX_GROUP"].map({1: 1, 2: 0})
df_filtered["SEX_LABEL"] = df_filtered["SEX"].map({1: "Laki-laki", 2: "Perempuan"})

selected_cols = [
    "SITE_ID", "RAW_FOLDER", "SUB_ID", "SUB_ID_STR", "FILE_ID",
    "DX_GROUP", "LABEL", "LABEL_BIN",
    "AGE_AT_SCAN", "SEX", "SEX_LABEL", "HANDEDNESS_CATEGORY",
    "FIQ", "VIQ", "PIQ",
    "HAS_RAW_FMRI", "HAS_RAW_SMRI"
]

df_clean = df_filtered[selected_cols].copy()
print(f"Dimensi data bersih: {df_clean.shape[0]} baris x {df_clean.shape[1]} kolom")
df_clean.head()

Dimensi data bersih: 1097 baris x 17 kolom


,SITE_ID,RAW_FOLDER,SUB_ID,SUB_ID_STR,FILE_ID,DX_GROUP,LABEL,LABEL_BIN,AGE_AT_SCAN,SEX,SEX_LABEL,HANDEDNESS_CATEGORY,FIQ,VIQ,PIQ,HAS_RAW_FMRI,HAS_RAW_SMRI
0,CALTECH,Caltech,51456,0051456,Caltech_0051456,1,ASD,1,55.4,1,Laki-laki,R,126.0,118.0,128.0,True,True
1,CALTECH,Caltech,51457,0051457,Caltech_0051457,1,ASD,1,22.9,1,Laki-laki,Ambi,107.0,119.0,93.0,True,True
2,CALTECH,Caltech,51458,0051458,Caltech_0051458,1,ASD,1,39.2,1,Laki-laki,R,93.0,80.0,108.0,True,True
3,CALTECH,Caltech,51459,0051459,Caltech_0051459,1,ASD,1,22.8,1,Laki-laki,R,106.0,94.0,118.0,True,True
4,CALTECH,Caltech,51460,0051460,Caltech_0051460,1,ASD,1,34.6,2,Perempuan,Ambi,133.0,135.0,122.0,True,True


## 5. Evaluasi Missing Value Sebelum Imputasi

In [6]:
missing_sebelum = pd.DataFrame({
    "Tipe_Data": df_clean.dtypes,
    "Jumlah_Kosong": df_clean.isna().sum(),
    "Persen_Kosong (%)": (df_clean.isna().mean() * 100).round(2)
})
missing_sebelum.to_csv(TABEL_DIR / "cleaning_missing_sebelum_imputasi.csv")
print("=== STATUS DATA KOSONG SEBELUM IMPUTASI ===")
display(missing_sebelum[missing_sebelum["Jumlah_Kosong"] > 0])

=== STATUS DATA KOSONG SEBELUM IMPUTASI ===


,Tipe_Data,Jumlah_Kosong,Persen_Kosong (%)
HANDEDNESS_CATEGORY,str,326,29.72
FIQ,float64,71,6.47
VIQ,float64,190,17.32
PIQ,float64,176,16.04


## 6. Eksekusi Imputasi Missing Values

1. **Skor IQ (`FIQ`, `VIQ`, `PIQ`)**: Imputasi menggunakan **Median per Situs (`SITE_ID`)** + fallback ke **Median Global**.
2. **Tangan Dominan (`HANDEDNESS_CATEGORY`)**: Imputasi menggunakan **Modus (*Mode*)**, yaitu nilai `'R'` (Tangan Kanan).

> **Catatan pencegahan data leakage.** Median per situs di sini dihitung dari **seluruh subjek**. Untuk keperluan eksplorasi dan tabel deskriptif hal ini tidak masalah, tetapi saat **melatih model**, statistik imputasi harus dihitung **hanya dari data latih di setiap fold** lalu diterapkan ke data uji. Karena itu, nilai asli sebelum imputasi tetap disimpan di kolom `FIQ_RAW`, `VIQ_RAW`, `PIQ_RAW`, dan `HANDEDNESS_RAW` pada berkas ekspor, supaya imputasi bisa diulang di dalam pipeline cross-validation. **Di alur akhir memang begitu:** notebook 09 mengambil `FIQ_RAW` (nilai asli, belum diimputasi), dan pengisian FIQ kosong dilakukan di dalam setiap fold pada notebook skenario S3–S4.

In [7]:
# Simpan nilai asli sebelum imputasi untuk perbandingan
df_clean["FIQ_RAW"] = df_clean["FIQ"]
df_clean["VIQ_RAW"] = df_clean["VIQ"]
df_clean["PIQ_RAW"] = df_clean["PIQ"]
df_clean["HANDEDNESS_RAW"] = df_clean["HANDEDNESS_CATEGORY"]

# 1. Imputasi Skor IQ (FIQ, VIQ, PIQ) menggunakan Median per Situs + Global Median Fallback
for col in ["FIQ", "VIQ", "PIQ"]:
    global_median = df_clean[col].median()
    site_medians = df_clean.groupby("SITE_ID")[col].transform("median")
    df_clean[col] = df_clean[col].fillna(site_medians).fillna(global_median).round(1)

# 2. Imputasi Tangan Dominan (HANDEDNESS_CATEGORY) menggunakan Modus (Mode)
modus_handedness = df_clean["HANDEDNESS_CATEGORY"].mode()[0] # Nilai 'R'
df_clean["HANDEDNESS_CATEGORY"] = df_clean["HANDEDNESS_CATEGORY"].fillna(modus_handedness)

print(f"Modus Handedness yang diterapkan: '{modus_handedness}' (Tangan Kanan)")
print("\n=== CONTOH HASIL IMPUTASI MISSING VALUES ===")
df_clean[["SITE_ID", "SUB_ID", "FIQ_RAW", "FIQ", "VIQ_RAW", "VIQ", "HANDEDNESS_RAW", "HANDEDNESS_CATEGORY"]].head(10)

Modus Handedness yang diterapkan: 'R' (Tangan Kanan)

=== CONTOH HASIL IMPUTASI MISSING VALUES ===


,SITE_ID,SUB_ID,FIQ_RAW,FIQ,VIQ_RAW,VIQ,HANDEDNESS_RAW,HANDEDNESS_CATEGORY
0,CALTECH,51456,126.0,126.0,118.0,118.0,R,R
1,CALTECH,51457,107.0,107.0,119.0,119.0,Ambi,Ambi
2,CALTECH,51458,93.0,93.0,80.0,80.0,R,R
3,CALTECH,51459,106.0,106.0,94.0,94.0,R,R
4,CALTECH,51460,133.0,133.0,135.0,135.0,Ambi,Ambi
5,CALTECH,51461,99.0,99.0,111.0,111.0,R,R
6,CALTECH,51462,107.0,107.0,102.0,102.0,R,R
7,CALTECH,51463,102.0,102.0,101.0,101.0,R,R
8,CALTECH,51464,101.0,101.0,118.0,118.0,Ambi,Ambi
9,CALTECH,51465,96.0,96.0,99.0,99.0,R,R


## 7. Verifikasi 0.00% Missing Value Pasca Imputasi

In [8]:
missing_sesudah = pd.DataFrame({
    "Tipe_Data": df_clean[selected_cols].dtypes,
    "Jumlah_Kosong": df_clean[selected_cols].isna().sum(),
    "Persen_Kosong (%)": (df_clean[selected_cols].isna().mean() * 100).round(2)
})
missing_sesudah.to_csv(TABEL_DIR / "cleaning_missing_sesudah_imputasi.csv")

print("=== VERIFIKASI: SELURUH 17 KOLOM KINI 0.00% MISSING VALUE ===")
display(missing_sesudah)

# Perbandingan Distribusi Handedness Sebelum vs Sesudah Imputasi Modus
komparasi_hand = pd.DataFrame({
    "Handedness Asli": df_clean["HANDEDNESS_RAW"].value_counts(dropna=False),
    "Handedness Setelah Imputasi Modus": df_clean["HANDEDNESS_CATEGORY"].value_counts(dropna=False)
})
komparasi_hand.to_csv(TABEL_DIR / "cleaning_komparasi_handedness_imputasi.csv")
print("\n=== PERBANDINGAN SEBARAN HANDEDNESS SEBELUM VS SESUDAH IMPUTASI MODUS ===")
display(komparasi_hand)

=== VERIFIKASI: SELURUH 17 KOLOM KINI 0.00% MISSING VALUE ===


,Tipe_Data,Jumlah_Kosong,Persen_Kosong (%)
SITE_ID,str,0,0.0
RAW_FOLDER,str,0,0.0
SUB_ID,int64,0,0.0
SUB_ID_STR,str,0,0.0
FILE_ID,str,0,0.0
DX_GROUP,int64,0,0.0
LABEL,str,0,0.0
LABEL_BIN,int64,0,0.0
AGE_AT_SCAN,float64,0,0.0
SEX,int64,0,0.0



=== PERBANDINGAN SEBARAN HANDEDNESS SEBELUM VS SESUDAH IMPUTASI MODUS ===


,Handedness Asli,Handedness Setelah Imputasi Modus
Ambi,15,15.0
L,71,71.0
L->R,1,1.0
Mixed,6,6.0
R,678,1004.0
NaN,326,NaN


## 8. Ekspor Dataset Fenotipe Bersih Siap Pakai (`phenotypic_clean.csv`)

Berkas ekspor berisi 17 kolom utama **ditambah** 4 kolom nilai asli sebelum imputasi (`*_RAW`) untuk imputasi ulang di dalam setiap fold.

In [9]:
# Ekspor 17 kolom utama + 4 kolom nilai asli sebelum imputasi (untuk imputasi di dalam fold)
kolom_raw = ["FIQ_RAW", "VIQ_RAW", "PIQ_RAW", "HANDEDNESS_RAW"]
df_clean_final = df_clean[selected_cols + kolom_raw].copy()
df_clean_final.to_csv(PHENO_CLEAN_PATH, index=False)

print(f"Dataset fenotipe bersih berhasil disimpan di: {PHENO_CLEAN_PATH}")
print(f"Total baris: {len(df_clean_final)} subjek, Total kolom: {len(df_clean_final.columns)} "
      f"({len(selected_cols)} kolom utama + {len(kolom_raw)} kolom nilai asli)")

Dataset fenotipe bersih berhasil disimpan di: C:\Users\hikma\Documents\Tugas Akhir\v7\data\phenotypic\phenotypic_clean.csv
Total baris: 1097 subjek, Total kolom: 21 (17 kolom utama + 4 kolom nilai asli)
